In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler 
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split 

In [2]:
# DATA INITIALIZATION
df = pd.read_csv(r'C:/Users/Inayatullah/Downloads/archive\\/WA_Fn-UseC_-Telco-Customer-Churn.csv')

print(df.head())

   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Contract Pape

In [3]:
# PHASE 1: DATA CLEANING & TYPE AUDIT
if "customerID" in df.columns:
    df.drop("customerID", axis=1, inplace=True)

In [4]:
# TotalCharges ko text se float me badla aur naye customers (blank spaces) ko 0 diya

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

df["TotalCharges"] = df["TotalCharges"].fillna(0)

In [5]:
# PHASE 2: CATEGORICAL FEATURE ENCODING

if df["Churn"].dtype == "object":
    df["Churn"] = df["Churn"].map({"Yes":1 , "No":0})

In [6]:
# Baqi saare text columns par ek sath One-Hot Encoding apply ki
# dtype=int se True/False ki jagah saaf-saaf 1 aur 0 milega

df_encoded = pd.get_dummies(df, drop_first=True, dtype=int)

In [8]:
# PHASE 3: STRATIFIED TRAIN-TEST SPLIT

x = df_encoded.drop(["Churn_Yes"], axis=1)

y = df_encoded["Churn_Yes"]

In [10]:
# stratify=y lagana zaroori hai taaki dono sets me Churn ka ratio barabar rahe

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

In [11]:
# PHASE 4: TARGETED FEATURE SCALING
scaler = StandardScaler()
numerical_features = ["tenure", "MonthlyCharges", "TotalCharges"]

In [12]:
x_train[numerical_features] = scaler.fit_transform(x_train[numerical_features])
x_test[numerical_features] = scaler.transform(x_test[numerical_features])

In [13]:
# PHASE 5: BALANCED MODEL TRAINING & EVALUATION

# class_weight='balanced' lagaya taaki imbalanced data par sahi results aein
model = RandomForestClassifier(
    n_estimators=100, max_depth=10, random_state=42, class_weight="balanced"
)

In [15]:
# Model fitting
model.fit(x_train, y_train)
print("Pipeline complete. Model trained successfully! 🚀")

Pipeline complete. Model trained successfully! 🚀


In [17]:
# Predictions and Results
y_predict = model.predict(x_test)

In [19]:
print("\n" + "=" * 50)
print("              INDUSTRY EVALUATION REPORT              ")
print("=" * 50)
print(classification_report(y_test, y_predict))
print("=" * 50)


              INDUSTRY EVALUATION REPORT              
              precision    recall  f1-score   support

           0       0.89      0.78      0.83      1035
           1       0.54      0.72      0.62       374

    accuracy                           0.76      1409
   macro avg       0.71      0.75      0.72      1409
weighted avg       0.79      0.76      0.77      1409

